# STAI-X 2026 — Final Submission (Award A)

**Self-contained. Run top-to-bottom on Kaggle.**

Outputs `../submission.csv` (repo root, relative to this notebook at `notebooks/`).

In [ ]:
import subprocess, sys

KAGGLE = False  # auto-detected below
try:
    from pathlib import Path
    if Path('/kaggle/input/staix-challenge').exists():
        KAGGLE = True
except:
    pass

if KAGGLE:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'lightgbm', 'xgboost', 'catboost', 'optuna', 'reportlab'])

print('Kaggle environment:', KAGGLE)

In [ ]:
import sys, os
# Works both locally (notebooks/ subdir) and on Kaggle (mounted at /kaggle/working)
nb_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)

import numpy as np
import pandas as pd
from src.predict import run
from pathlib import Path

DATA_ROOT   = Path('/kaggle/input/staix-challenge') if KAGGLE else None
OUTPUT_PATH = Path('/kaggle/working/submission.csv') if KAGGLE else Path('../submission.csv')

print('Data root:', DATA_ROOT)
print('Output:   ', OUTPUT_PATH)

In [ ]:
submission = run(data_root=DATA_ROOT, output_path=OUTPUT_PATH, verbose=True)

In [ ]:
# Validation checks
print('Rows:', len(submission))
print('NaN:', submission['rate_per_10000_ed_visits'].isna().sum())
print('Negative:', (submission['rate_per_10000_ed_visits'] < 0).sum())
print('Min:', submission['rate_per_10000_ed_visits'].min())
print('Max:', submission['rate_per_10000_ed_visits'].max())
print('\nHead:')
submission.head(10)

In [ ]:
# Schema portability test — simulate a new unseen period_id
from src.data_loader import load_submission_template, load_val
template = load_submission_template(DATA_ROOT)

# Confirm every row_id from template is present
assert set(submission['row_id']) == set(template['row_id']), 'row_id mismatch'
assert list(submission.columns) == ['row_id', 'rate_per_10000_ed_visits'], 'Column schema wrong'
print('Schema checks passed. Ready to submit.')